# La Torre · Práctica 3.1 — Entrenar el bot en CPU y en GPU

Entrena con PPO al caballero contra el Guardián de Hierro **con la misma configuración** que en la PC local, una vez en CPU y otra en GPU, y mide tiempos.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.

**Capturas para el entregable:** la salida de la celda 1 (GPU asignada) y la tabla de la última celda.

In [ ]:
# 1. ¿Qué GPU y CPU nos asignó Colab?
!nvidia-smi --query-gpu=name,memory.total --format=csv
!lscpu | grep -E 'Model name|^CPU\(s\)'

In [ ]:
# 2. Sube ia/colab/la-torre-ia.zip (se genera en tu PC con: python ia/empaquetar_colab.py)
from google.colab import files
subido = files.upload()
!rm -rf la-torre && mkdir la-torre && unzip -q -o la-torre-ia.zip -d la-torre && ls la-torre

In [ ]:
# 3. Librerías (PyTorch ya viene instalado en Colab)
!pip install -q gymnasium stable-baselines3
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 4. Misma configuración que en la PC: 100 000 pasos, 1 entorno
%cd /content/la-torre
!python ia/entrenar.py --device cpu  --pasos 100000 --nombre colab_cpu_100k
!python ia/entrenar.py --device cuda --pasos 100000 --nombre colab_gpu_100k

In [ ]:
# 5. (Opcional) 8 entornos en paralelo: ¿mejora la GPU con más datos por lote?
!python ia/entrenar.py --device cpu  --pasos 200000 --entornos 8 --nombre colab_cpu_200k_8env
!python ia/entrenar.py --device cuda --pasos 200000 --entornos 8 --nombre colab_gpu_200k_8env

In [ ]:
# 6. Latencia de inferencia en la CPU de Colab y evaluación del modelo entrenado en GPU
!python ia/medir.py ia/modelos/colab_gpu_100k.zip --episodios 30

In [ ]:
# 7. Tabla de resultados (captura) y descarga para subirlos al repositorio
import json, pandas as pd
filas = [json.loads(l) for l in open('ia/resultados/entrenamiento.jsonl', encoding='utf-8')]
display(pd.DataFrame(filas)[['nombre', 'device', 'gpu', 'pasos', 'entornos', 'segundos', 'pasos_por_s']])
!cd ia && zip -q -r /content/resultados_colab.zip resultados
files.download('/content/resultados_colab.zip')